# 04 V2 - Tracking Evaluation

Obiettivo:
- confrontare le tracce prodotte dal tracker gerarchico con il ground truth SportsMOT;
- valutare il tracking dopo la rimozione degli arbitri;
- convertire le predizioni nel formato MOTChallenge;
- utilizzare TrackEval;
- calcolare metriche quantitative di tracking.

Pipeline valutata:

RF-DETR Nano fine-tuned  
→ geometric association + Kalman filter  
→ selective Re-ID OSNet  
→ SAM2 nei casi ambigui  
→ referee filtering  
→ TrackEval

Metriche principali:
- HOTA
- DetA
- AssA
- IDF1
- MOTA
- ID switches
- Fragmentation

In [1]:
from pathlib import Path
import os

PROJECT_ROOT = Path(
    "/Users/marcodalbis/Desktop/Tesi Magistrale/"
    "basketball-thesis/basketball-thesis"
)

TRACKEVAL_ROOT = PROJECT_ROOT / "TrackEval"

print("TrackEval root:", TRACKEVAL_ROOT)
print("Exists:", TRACKEVAL_ROOT.exists())

TrackEval root: /Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval
Exists: True


In [3]:
from pathlib import Path
import pandas as pd
import shutil
import os

DRIVE_ROOT = Path(
    "/Users/marcodalbis/Library/CloudStorage/"
    "GoogleDrive-marco.dalbis@gmail.com/"
    "Il mio Drive/"
    "basketball-thesis"
)

SPORTSMOT_ROOT = (
    DRIVE_ROOT
    / "datasets"
    / "sportsmot"
)

TRACKING_OUTPUT_ROOT = (
    DRIVE_ROOT
    / "experiments"
    / "hierarchical_tracking"
)

print("SportsMOT:", SPORTSMOT_ROOT)
print("Tracking outputs:", TRACKING_OUTPUT_ROOT)
print("TrackEval:", TRACKEVAL_ROOT)

SportsMOT: /Users/marcodalbis/Library/CloudStorage/GoogleDrive-marco.dalbis@gmail.com/Il mio Drive/basketball-thesis/datasets/sportsmot
Tracking outputs: /Users/marcodalbis/Library/CloudStorage/GoogleDrive-marco.dalbis@gmail.com/Il mio Drive/basketball-thesis/experiments/hierarchical_tracking
TrackEval: /Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval


Sequenza da analizzare

In [4]:
SEQUENCE_NAME = "v_00HRwkvvjtQ_c001"

GT_SEQUENCE_DIR = (
    SPORTSMOT_ROOT
    / "val"
    / SEQUENCE_NAME
)

GT_FILE = (
    GT_SEQUENCE_DIR
    / "gt"
    / "gt.txt"
)

PREDICTIONS_CSV = (
    TRACKING_OUTPUT_ROOT
    / f"{SEQUENCE_NAME}_hierarchical_players_only.csv"
)

print("GT:", GT_FILE)
print("Predizioni:", PREDICTIONS_CSV)

print("GT esiste:", GT_FILE.exists())
print("Predizioni esistono:", PREDICTIONS_CSV.exists())

assert GT_FILE.exists(), GT_FILE
assert PREDICTIONS_CSV.exists(), PREDICTIONS_CSV

GT: /Users/marcodalbis/Library/CloudStorage/GoogleDrive-marco.dalbis@gmail.com/Il mio Drive/basketball-thesis/datasets/sportsmot/val/v_00HRwkvvjtQ_c001/gt/gt.txt
Predizioni: /Users/marcodalbis/Library/CloudStorage/GoogleDrive-marco.dalbis@gmail.com/Il mio Drive/basketball-thesis/experiments/hierarchical_tracking/v_00HRwkvvjtQ_c001_hierarchical_players_only.csv
GT esiste: True
Predizioni esistono: True


In [5]:
gt_df = pd.read_csv(
    GT_FILE,
    header=None
)

print("Shape GT:", gt_df.shape)

gt_df.head()

Shape GT: (11276, 9)


,0,1,2,3,4,5,6,7,8
0,1,0,438,269,118,169,1,1,1
1,1,1,973,389,78,216,1,1,1
2,1,2,634,429,95,198,1,1,1
3,1,3,649,272,108,152,1,1,1
4,1,4,637,421,129,159,1,1,1


In [6]:
print("Numero frame GT:", gt_df[0].nunique())
print("Numero ID GT:", gt_df[1].nunique())

print("Frame minimo:", gt_df[0].min())
print("Frame massimo:", gt_df[0].max())

Numero frame GT: 1162
Numero ID GT: 10
Frame minimo: 1
Frame massimo: 1162


In [8]:
pred_df = pd.read_csv(
    PREDICTIONS_CSV
)

print("Shape predictions:", pred_df.shape)
print("Columns:")
print(pred_df.columns.tolist())

pred_df.head()

Shape predictions: (11318, 10)
Columns:
['frame', 'track_id', 'x1', 'y1', 'x2', 'y2', 'width', 'height', 'confidence', 'confirmed']


,frame,track_id,x1,y1,x2,y2,width,height,confidence,confirmed
0,1,0,439.375305,266.329010,558.552917,436.577148,119.177612,170.248138,0.938461,False
1,1,1,971.564453,389.879883,1049.835327,605.411072,78.270874,215.531189,0.915086,False
2,1,2,637.455872,419.139099,764.137878,627.321350,126.682007,208.182251,0.870535,False
3,1,3,990.353394,290.022308,1052.396118,495.324768,62.042725,205.302460,0.847963,False
4,1,4,644.804443,272.780945,756.591858,430.809967,111.787415,158.029022,0.823956,False


In [9]:
print("Predizioni:")
print("Frame minimo:", pred_df["frame"].min())
print("Frame massimo:", pred_df["frame"].max())
print("Numero frame:", pred_df["frame"].nunique())

print()

print("Ground truth:")
print("Frame minimo:", gt_df[0].min())
print("Frame massimo:", gt_df[0].max())
print("Numero frame:", gt_df[0].nunique())

Predizioni:
Frame minimo: 1
Frame massimo: 1162
Numero frame: 1162

Ground truth:
Frame minimo: 1
Frame massimo: 1162
Numero frame: 1162


## Controllo tracker

In [36]:
print(
    "Tracker ID invalidi (-1):",
    (pred_df["track_id"] == -1).sum()
)

print(
    "Tracker ID unici:",
    pred_df["track_id"].nunique()
)

Tracker ID invalidi (-1): 0
Tracker ID unici: 95


Conversione delle bbox nel formato MOTChallenge

In [37]:
mot_pred = pd.DataFrame()

mot_pred["frame"] = pred_df["frame"].astype(int)
mot_pred["id"] = pred_df["track_id"].astype(int)

mot_pred["bb_left"] = pred_df["x1"]
mot_pred["bb_top"] = pred_df["y1"]

mot_pred["bb_width"] = (
    pred_df["x2"] - pred_df["x1"]
)

mot_pred["bb_height"] = (
    pred_df["y2"] - pred_df["y1"]
)

mot_pred["conf"] = pred_df["confidence"]

mot_pred["x"] = -1
mot_pred["y"] = -1
mot_pred["z"] = -1

mot_pred.head()

,frame,id,bb_left,bb_top,bb_width,bb_height,conf,x,y,z
0,1,0,439.375305,266.329010,119.177612,170.248138,0.938461,-1,-1,-1
1,1,1,971.564453,389.879883,78.270874,215.531189,0.915086,-1,-1,-1
2,1,2,637.455872,419.139099,126.682007,208.182251,0.870535,-1,-1,-1
3,1,3,990.353394,290.022308,62.042725,205.302460,0.847963,-1,-1,-1
4,1,4,644.804443,272.780945,111.787415,158.029022,0.823956,-1,-1,-1


In [38]:
# Ordinamento MOTChallenge
mot_pred = mot_pred.sort_values(
    ["frame", "id"]
).reset_index(drop=True)

# Controlli di consistenza
assert mot_pred.shape[1] == 10, (
    f"Numero colonne errato: {mot_pred.shape[1]}"
)

assert not mot_pred.isna().any().any(), (
    "Sono presenti valori NaN nelle predizioni."
)

assert (mot_pred["frame"] >= 1).all()
assert (mot_pred["id"] >= 0).all()

print("Numero righe:", len(mot_pred))
print("Numero colonne:", mot_pred.shape[1])
print("Frame:", mot_pred["frame"].min(), "→", mot_pred["frame"].max())
print("Track ID unici:", mot_pred["id"].nunique())

mot_pred.head()

Numero righe: 11318
Numero colonne: 10
Frame: 1 → 1162
Track ID unici: 95


,frame,id,bb_left,bb_top,bb_width,bb_height,conf,x,y,z
0,1,0,439.375305,266.329010,119.177612,170.248138,0.938461,-1,-1,-1
1,1,1,971.564453,389.879883,78.270874,215.531189,0.915086,-1,-1,-1
2,1,2,637.455872,419.139099,126.682007,208.182251,0.870535,-1,-1,-1
3,1,3,990.353394,290.022308,62.042725,205.302460,0.847963,-1,-1,-1
4,1,4,644.804443,272.780945,111.787415,158.029022,0.823956,-1,-1,-1


Controllo bbox invalide

In [39]:
invalid_boxes = mot_pred[
    (mot_pred["bb_width"] <= 0)
    | (mot_pred["bb_height"] <= 0)
    | mot_pred["bb_left"].isna()
    | mot_pred["bb_top"].isna()
    | mot_pred["conf"].isna()
]

print("Bounding box invalide:", len(invalid_boxes))

invalid_boxes.head()

Bounding box invalide: 0


,frame,id,bb_left,bb_top,bb_width,bb_height,conf,x,y,z


In [40]:
assert len(invalid_boxes) == 0, (
    "Sono presenti bounding box invalide."
)

Preparazione struttura TrackEval

In [41]:
BENCHMARK = "SportsMOT"
SPLIT = "val"

TRACKER_NAME = "HierarchicalTracker_PlayersOnly"

GT_ROOT = (
    TRACKEVAL_ROOT
    / "data"
    / "gt"
    / "mot_challenge"
    / f"{BENCHMARK}-{SPLIT}"
)

TRACKER_ROOT = (
    TRACKEVAL_ROOT
    / "data"
    / "trackers"
    / "mot_challenge"
    / f"{BENCHMARK}-{SPLIT}"
    / TRACKER_NAME
    / "data"
)

GT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

TRACKER_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print("GT root:")
print(GT_ROOT)

print("\nTracker root:")
print(TRACKER_ROOT)

GT root:
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/gt/mot_challenge/SportsMOT-val

Tracker root:
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/HierarchicalTracker_PlayersOnly/data


## Copia del ground truth

sequence/
├── gt/
│   └── gt.txt
└── seqinfo.ini

In [23]:
target_sequence_dir = GT_ROOT / SEQUENCE_NAME
target_gt_dir = target_sequence_dir / "gt"

target_gt_dir.mkdir(parents=True, exist_ok=True)

shutil.copy(
    GT_SEQUENCE_DIR / "gt" / "gt.txt",
    target_gt_dir / "gt.txt"
)

shutil.copy(
    GT_SEQUENCE_DIR / "seqinfo.ini",
    target_sequence_dir / "seqinfo.ini"
)

print("GT copiato.")

GT copiato.


In [24]:
TRACKER_FILE = TRACKER_ROOT / f"{SEQUENCE_NAME}.txt"

mot_pred.to_csv(
    TRACKER_FILE,
    index=False,
    header=False,
)

print("Predizioni salvate in:")
print(TRACKER_FILE)

Predizioni salvate in:
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/HierarchicalTracker_PlayersOnly/data/v_00HRwkvvjtQ_c001.txt


creazione seqmap

In [25]:
SEQMAP_DIR = (
    TRACKEVAL_ROOT
    / "data"
    / "gt"
    / "mot_challenge"
    / "seqmaps"
)

SEQMAP_DIR.mkdir(parents=True, exist_ok=True)

SEQMAP_FILE = SEQMAP_DIR / f"{BENCHMARK}-{SPLIT}.txt"

with open(SEQMAP_FILE, "w") as f:
    f.write("name\n")
    f.write(f"{SEQUENCE_NAME}\n")

print(SEQMAP_FILE.read_text())

name
v_00HRwkvvjtQ_c001



controllo sequenza finale

In [26]:
print("Ground truth:")
print(GT_ROOT)

print("\nTracker:")
print(TRACKER_ROOT)

print("\nSeqmap:")
print(SEQMAP_FILE)

print("\nPredizione:")
print(TRACKER_FILE)

Ground truth:
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/gt/mot_challenge/SportsMOT-val

Tracker:
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/HierarchicalTracker_PlayersOnly/data

Seqmap:
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/gt/mot_challenge/seqmaps/SportsMOT-val.txt

Predizione:
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/HierarchicalTracker_PlayersOnly/data/v_00HRwkvvjtQ_c001.txt


In [27]:
for path in (
    TRACKEVAL_ROOT
    / "data"
    / "gt"
    / "mot_challenge"
    / "SportsMOT-val"
).rglob("*"):
    if path.is_file():
        print(path)

/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/gt/mot_challenge/SportsMOT-val/v_00HRwkvvjtQ_c001/seqinfo.ini
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/gt/mot_challenge/SportsMOT-val/v_00HRwkvvjtQ_c001/gt/gt.txt


In [28]:
for path in (
    TRACKEVAL_ROOT
    / "data"
    / "trackers"
    / "mot_challenge"
    / "SportsMOT-val"
).rglob("*"):
    if path.is_file():
        print(path)

/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/RFDETR_SAM2_CLEAN15/pedestrian_summary.txt
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/RFDETR_SAM2_CLEAN15/pedestrian_detailed.csv
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/RFDETR_SAM2_CLEAN15/pedestrian_plot.pdf
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/RFDETR_SAM2_CLEAN15/pedestrian_plot.png
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/RFDETR_SAM2_CLEAN15/data/v_00HRwkvvjtQ_c001.txt
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val

## Prima valutazione TrackEval

In [29]:
import subprocess

cmd = [
    "python",
    "scripts/run_mot_challenge.py",
    "--BENCHMARK", "SportsMOT",
    "--SPLIT_TO_EVAL", "val",
    "--TRACKERS_TO_EVAL", TRACKER_NAME,
    "--METRICS", "HOTA", "CLEAR", "Identity",
    "--USE_PARALLEL", "False",
    "--DO_PREPROC", "False",
]

result = subprocess.run(
    cmd,
    cwd=TRACKEVAL_ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

print("Return code:", result.returncode)


Eval Config:
USE_PARALLEL         : False                         
NUM_PARALLEL_CORES   : 8                             
BREAK_ON_ERROR       : True                          
RETURN_ON_ERROR      : False                         
LOG_ON_ERROR         : /Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/error_log.txt
PRINT_RESULTS        : True                          
PRINT_ONLY_COMBINED  : False                         
PRINT_CONFIG         : True                          
TIME_PROGRESS        : True                          
DISPLAY_LESS_PROGRESS : False                         
OUTPUT_SUMMARY       : True                          
OUTPUT_EMPTY_CLASSES : True                          
OUTPUT_DETAILED      : True                          
PLOT_CURVES          : True                          

MotChallenge2DBox Config:
PRINT_CONFIG         : True                          
GT_FOLDER            : /Users/marcodalbis/Desktop/Tesi Magistrale/basketball

# Risultati Tracking Evaluation

Sequenza valutata:

`v_00HRwkvvjtQ_c001`

Pipeline:

RF-DETR Nano fine-tuned  
→ geometric association + Kalman filter  
→ selective Re-ID OSNet  
→ SAM2 local ambiguity handling  
→ referee filtering  
→ TrackEval

Metriche principali:

- HOTA:
- DetA:
- AssA:
- LocA:
- IDF1:
- IDP:
- IDR:
- MOTA:
- MOTP:
- ID switches:
- Fragmentation:

## Interpretazione preliminare

Da compilare dopo l'esecuzione:

- qualità complessiva del tracking;
- qualità della detection;
- qualità dell'associazione temporale;
- stabilità degli ID;
- numero di ID switch;
- frammentazione delle traiettorie;
- comportamento durante occlusioni e contatti ravvicinati;
- effetto del filtraggio degli arbitri.

## Conclusione

Da compilare dopo l'analisi delle metriche.

Individuazione file risultato

In [30]:
result_files = list(
    (
        TRACKEVAL_ROOT
        / "data"
        / "trackers"
        / "mot_challenge"
        / f"{BENCHMARK}-{SPLIT}"
        / TRACKER_NAME
    ).rglob("*")
)

for f in result_files:
    if f.is_file():
        print(f)

/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/HierarchicalTracker_PlayersOnly/pedestrian_summary.txt
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/HierarchicalTracker_PlayersOnly/pedestrian_detailed.csv
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/HierarchicalTracker_PlayersOnly/pedestrian_plot.pdf
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/HierarchicalTracker_PlayersOnly/pedestrian_plot.png
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/HierarchicalTracker_PlayersOnly/data/v_00HRwkvvjtQ_c001.txt


Lettura summary

In [31]:
summary_files = [
    f for f in result_files
    if "summary" in f.name.lower()
]

summary_files

[PosixPath('/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/HierarchicalTracker_PlayersOnly/pedestrian_summary.txt')]

In [32]:
for summary_file in summary_files:
    print("\nFILE:", summary_file)
    print(summary_file.read_text())


FILE: /Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/HierarchicalTracker_PlayersOnly/pedestrian_summary.txt
HOTA DetA AssA DetRe DetPr AssRe AssPr LocA OWTA HOTA(0) LocA(0) HOTALocA(0) MOTA MOTP MODA CLR_Re CLR_Pr MTR PTR MLR CLR_TP CLR_FN CLR_FP IDSW MT PT ML Frag sMOTA IDF1 IDR IDP IDTP IDFN IDFP Dets GT_Dets IDs GT_IDs
48.195 78.09 29.848 84.14 83.827 32.443 72.861 86.379 50.112 57.54 83.755 48.192 93.207 84.58 94.484 97.428 97.067 100 0 0 10986 290 332 144 10 0 0 120 78.183 46.048 46.133 45.962 5202 6074 6116 11318 11276 95 10



In [33]:
summary_tables = []

for summary_file in summary_files:

    try:
        df_summary = pd.read_csv(
            summary_file,
            sep=r"\s+"
        )

        df_summary["source_file"] = summary_file.name

        summary_tables.append(df_summary)

    except Exception as e:
        print(
            "Errore leggendo",
            summary_file,
            ":",
            e
        )

Salvataggio risultati sul Drive

In [34]:
EVAL_OUTPUT_DIR = (
    DRIVE_ROOT
    / "experiments"
    / "hierarchical_tracking"
    / "evaluation_players_only"
    / SEQUENCE_NAME
)

EVAL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TRACKER_RESULT_DIR = (
    TRACKEVAL_ROOT
    / "data"
    / "trackers"
    / "mot_challenge"
    / f"{BENCHMARK}-{SPLIT}"
    / TRACKER_NAME
)

shutil.copytree(
    TRACKER_RESULT_DIR,
    EVAL_OUTPUT_DIR,
    dirs_exist_ok=True,
)

print("Risultati copiati in:")
print(EVAL_OUTPUT_DIR)

Risultati copiati in:
/Users/marcodalbis/Library/CloudStorage/GoogleDrive-marco.dalbis@gmail.com/Il mio Drive/basketball-thesis/experiments/hierarchical_tracking/evaluation_players_only/v_00HRwkvvjtQ_c001


In [35]:
if summary_tables:

    tracking_metrics_df = pd.concat(
        summary_tables,
        ignore_index=True
    )

    display(tracking_metrics_df)

else:
    tracking_metrics_df = pd.DataFrame()

    print("Nessun summary leggibile trovato.")

,HOTA,DetA,AssA,DetRe,DetPr,AssRe,AssPr,LocA,OWTA,HOTA(0),...,IDR,IDP,IDTP,IDFN,IDFP,Dets,GT_Dets,IDs,GT_IDs,source_file
0,48.195,78.09,29.848,84.14,83.827,32.443,72.861,86.379,50.112,57.54,...,46.133,45.962,5202,6074,6116,11318,11276,95,10,pedestrian_summary.txt


# Risultati Tracking Evaluation

## Sequenza valutata

`v_00HRwkvvjtQ_c001`

La sequenza contiene:

- **1162 frame**
- **10 identità ground-truth**
- **11276 annotazioni ground-truth**
- **11318 detection prodotte dal tracker**
- **95 identità predette dopo il filtraggio degli arbitri**

---

## Pipeline valutata

RF-DETR Nano fine-tuned  
→ geometric association  
→ Kalman filtering  
→ selective OSNet Re-Identification  
→ SAM2 local ambiguity handling  
→ referee filtering  
→ TrackEval

---

## Metriche principali

| Metric | Value |
|---|---:|
| HOTA | 48.195 |
| DetA | 78.090 |
| AssA | 29.848 |
| LocA | 86.379 |
| MOTA | 93.207 |
| MOTP | 84.580 |
| IDF1 | 46.048 |
| ID Recall | 46.133 |
| ID Precision | 45.962 |
| Detection Recall | 97.428 |
| Detection Precision | 97.067 |
| ID Switches | 144 |
| Fragmentations | 120 |
| False Negatives | 290 |
| False Positives | 332 |

---

## Interpretazione

I risultati mostrano una forte differenza tra la qualità della detection e
la qualità dell'associazione temporale.

Il tracker raggiunge valori elevati di detection recall e precision,
rispettivamente pari a circa 97.4% e 97.1%, e una MOTA pari a 93.2%.

Questo indica che il sistema è generalmente in grado di localizzare
correttamente i giocatori nei frame.

La principale criticità riguarda invece il mantenimento dell'identità nel
tempo. L'Association Accuracy (AssA) è pari a 29.85% e l'IDF1 a 46.05%.

Il sistema produce inoltre 95 identità differenti rispetto alle 10 identità
presenti nel ground truth, con 144 identity switches e 120 frammentazioni.

Questo comportamento è coerente con l'analisi qualitativa del video:
le bounding box rimangono generalmente corrette, mentre occlusioni,
interazioni ravvicinate e uscite/rientri dal campo visivo possono causare
la perdita o la riassegnazione dell'identità.

---

## Conclusione

Il sistema proposto mostra buone prestazioni nella localizzazione dei
giocatori, ma prestazioni più limitate nella long-term identity association.

Questi risultati evidenziano che il principale margine di miglioramento
non riguarda il detector RF-DETR, ma i moduli di associazione e
Re-Identification utilizzati per mantenere un'identità persistente nel tempo.

In [42]:
# ============================================================
# FINAL METRICS TABLE
# ============================================================

final_metrics = pd.DataFrame({
    "Metric": [
        "HOTA",
        "DetA",
        "AssA",
        "LocA",
        "MOTA",
        "MOTP",
        "IDF1",
        "Detection Recall",
        "Detection Precision",
        "ID Recall",
        "ID Precision",
        "ID Switches",
        "Fragmentations",
        "False Positives",
        "False Negatives",
        "Predicted IDs",
        "Ground Truth IDs",
    ],
    "Value": [
        48.195,
        78.090,
        29.848,
        86.379,
        93.207,
        84.580,
        46.048,
        97.428,
        97.067,
        46.133,
        45.962,
        144,
        120,
        332,
        290,
        95,
        10,
    ]
})

display(final_metrics)

,Metric,Value
0,HOTA,48.195
1,DetA,78.090
2,AssA,29.848
3,LocA,86.379
4,MOTA,93.207
5,MOTP,84.580
6,IDF1,46.048
7,Detection Recall,97.428
8,Detection Precision,97.067
9,ID Recall,46.133


In [43]:
FINAL_METRICS_PATH = (
    EVAL_OUTPUT_DIR
    / "tracking_metrics_summary.csv"
)

final_metrics.to_csv(
    FINAL_METRICS_PATH,
    index=False
)

print("Metrics saved:")
print(FINAL_METRICS_PATH)

Metrics saved:
/Users/marcodalbis/Library/CloudStorage/GoogleDrive-marco.dalbis@gmail.com/Il mio Drive/basketball-thesis/experiments/hierarchical_tracking/evaluation_players_only/v_00HRwkvvjtQ_c001/tracking_metrics_summary.csv
